# Projeto 2 - Cena com Gazebo


Nome: Pedro Louro Fernandes

NUSP: 13672446

In [19]:
import math
import os
import ctypes
import glfw
import numpy as np
from OpenGL.GL import *
from PIL import Image

SHADER_VS = """
#version 330 core
in vec3 posicao;
in vec2 texcoord;

uniform mat4 modelo;
uniform mat4 visao;
uniform mat4 projecao;

out vec2 v_tex;

void main() {
    gl_Position = projecao * visao * modelo * vec4(posicao, 1.0);
    v_tex = texcoord;
}
"""

SHADER_FS = """
#version 330 core
in vec2 v_tex;

uniform vec4 cor;
uniform sampler2D tex;
uniform int usa_textura;

out vec4 fragColor;

void main() {
    if (usa_textura == 1) {
        fragColor = texture(tex, v_tex);
    } else {
        fragColor = cor;
    }
}
"""

DEFAULT_COLOR = (0.8, 0.8, 0.8, 1.0)


## Funcoes de matriz

Este bloco concentra a base matematica de transformacoes 3D usada em toda a cena.

Funcoes implementadas:
- `ident`: matriz identidade 4x4;
- `transl`: translacao nos eixos X, Y, Z;
- `esc`: escala uniforme ou por eixo;
- `rot_x`, `rot_y`, `rot_z`: rotacoes em torno de cada eixo;
- `perspectiva`: projecao perspectiva com fovy, aspect ratio, near e far;
- `look_at`: constroi a matriz de visao a partir de posicao, alvo e vetor up.

Todas as matrizes sao 4x4 em float32 e compostas via multiplicacao (`@`) na funcao de desenho.


In [20]:
def ident():
    return np.eye(4, dtype=np.float32)

def transl(tx, ty, tz):
    m = ident()
    m[0, 3], m[1, 3], m[2, 3] = tx, ty, tz
    return m

def esc(sx, sy, sz):
    m = ident()
    m[0, 0], m[1, 1], m[2, 2] = sx, sy, sz
    return m

def rot_x(a):
    c, s = math.cos(a), math.sin(a)
    m = ident()
    m[1, 1], m[1, 2], m[2, 1], m[2, 2] = c, -s, s, c
    return m

def rot_y(a):
    c, s = math.cos(a), math.sin(a)
    m = ident()
    m[0, 0], m[0, 2], m[2, 0], m[2, 2] = c, s, -s, c
    return m

def rot_z(a):
    c, s = math.cos(a), math.sin(a)
    m = ident()
    m[0, 0], m[0, 1], m[1, 0], m[1, 1] = c, -s, s, c
    return m

def perspectiva(fovy, asp, near, far):
    f = 1.0 / math.tan(fovy / 2.0)
    m = np.zeros((4, 4), dtype=np.float32)
    m[0, 0], m[1, 1] = f / asp, f
    m[2, 2], m[2, 3], m[3, 2] = (far + near) / (near - far), (2.0 * far * near) / (near - far), -1.0
    return m

def look_at(eye, target, up=(0.0, 1.0, 0.0)):
    f = target - eye
    f = f / np.linalg.norm(f)
    u = np.array(up, dtype=np.float32)
    u = u / np.linalg.norm(u)
    s = np.cross(f, u)
    s = s / np.linalg.norm(s)
    u = np.cross(s, f)

    m = ident()
    m[0, 0], m[0, 1], m[0, 2] = s[0], s[1], s[2]
    m[1, 0], m[1, 1], m[1, 2] = u[0], u[1], u[2]
    m[2, 0], m[2, 1], m[2, 2] = -f[0], -f[1], -f[2]
    return m @ transl(-eye[0], -eye[1], -eye[2])


## Carregamento de assets e skybox

Este bloco reune as funcoes responsaveis por ler dados de disco e prepara-los para a GPU.

- `ler_mtl`: le arquivo .mtl e extrai cor difusa (Kd) e mapa de textura (map_Kd) por material;
- `carregar_textura`: carrega imagem com PIL, envia para a GPU como textura 2D e gera mipmaps.
  O parametro `clamp=True` usa `GL_CLAMP_TO_EDGE`, necessario para a skybox evitar costuras visiveis;
- `preparar_materiais`: resolve todos os materiais de um .obj, carregando texturas quando disponiveis
  e aplicando fallback de cor difusa caso contrario. Para a lanterna, usa `lantern_Base_Color.jpg`
  como textura padrao se nenhuma for especificada no .mtl;
- `carregar_obj`: parser de .obj que suporta vertices, coordenadas UV, faces poligonais (trianguladas
  em leque) e multiplos materiais via `usemtl`. Retorna vertices interleaved (x,y,z,u,v),
  lista de batches por material, e bounding box da cena;
- `skybox_vertices`: gera uma esfera equiretangular (16 stacks x 32 slices) com UV continuo,
  eliminando os artefatos de costura que surgiam com o cubo original.


In [21]:
def ler_mtl(caminho):
    materiais = {}
    atual = None
    if not os.path.exists(caminho):
        return materiais

    with open(caminho, "r", encoding="utf-8", errors="ignore") as arq:
        for linha in arq:
            linha = linha.strip()
            if not linha or linha.startswith("#"):
                continue

            chave, *resto = linha.split(maxsplit=1)
            valor = resto[0] if resto else ""

            if chave == "newmtl":
                atual = valor
                materiais[atual] = {}
            elif atual is not None and chave == "Kd":
                partes = [float(x) for x in valor.split()[:3]]
                materiais[atual]["Kd"] = (partes[0], partes[1], partes[2])
            elif atual is not None and chave == "map_Kd":
                materiais[atual]["map_Kd"] = valor

    return materiais

def carregar_textura(caminho, clamp=False):
    if not os.path.exists(caminho):
        return 0
    img = Image.open(caminho).convert("RGBA")
    if not clamp:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    dados = img.tobytes()
    wrap = GL_CLAMP_TO_EDGE if clamp else GL_REPEAT
    tex_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, wrap)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, wrap)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, img.width, img.height, 0, GL_RGBA, GL_UNSIGNED_BYTE, dados)
    glGenerateMipmap(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, 0)
    return tex_id

def preparar_materiais(materiais, base_dir):
    saida = {}
    for nome, mat in materiais.items():
        cor = mat.get("Kd", (1.0, 1.0, 1.0))
        tex = 0
        mapa = mat.get("map_Kd")
        if not mapa and os.path.basename(base_dir).lower() == "lanterna":
            fallback_tex = "lantern_Base_Color.jpg"
            if os.path.exists(os.path.join(base_dir, fallback_tex)):
                mapa = fallback_tex
        if mapa:
            tex = carregar_textura(os.path.join(base_dir, mapa))
        saida[nome] = {"cor": (cor[0], cor[1], cor[2], 1.0), "tex": tex}
    return saida

def carregar_obj(caminho):
    base_dir = os.path.dirname(caminho)
    materiais = {}
    vertices = []
    uvs = []
    dados = {}
    atual = "__default__"
    dados[atual] = []

    def parse_indice(valor, tamanho):
        idx = int(valor)
        return idx - 1 if idx > 0 else tamanho + idx

    with open(caminho, "r", encoding="utf-8", errors="ignore") as arq:
        for linha in arq:
            linha = linha.strip()
            if not linha or linha.startswith("#"):
                continue

            partes = linha.split()
            cmd = partes[0]

            if cmd == "v":
                vertices.append((float(partes[1]), float(partes[2]), float(partes[3])))
            elif cmd == "vt":
                uvs.append((float(partes[1]), float(partes[2])))
            elif cmd == "mtllib":
                mtl_path = os.path.join(base_dir, " ".join(partes[1:]))
                materiais.update(ler_mtl(mtl_path))
            elif cmd == "usemtl":
                atual = " ".join(partes[1:])
                dados.setdefault(atual, [])
            elif cmd == "f":
                indices = []
                for tok in partes[1:]:
                    campos = tok.split("/")
                    vi = parse_indice(campos[0], len(vertices))
                    vti = None
                    if len(campos) > 1 and campos[1]:
                        vti = parse_indice(campos[1], len(uvs))
                    indices.append((vi, vti))

                for i in range(1, len(indices) - 1):
                    for vi, vti in (indices[0], indices[i], indices[i + 1]):
                        px, py, pz = vertices[vi]
                        if vti is not None and vti < len(uvs):
                            tu, tv = uvs[vti]
                        else:
                            tu, tv = 0.0, 0.0
                        dados[atual].extend([px, py, pz, tu, tv])

    if vertices:
        arr = np.array(vertices, dtype=np.float32)
        min_v = np.min(arr, axis=0)
        max_v = np.max(arr, axis=0)
    else:
        min_v = np.array([0.0, 0.0, 0.0], dtype=np.float32)
        max_v = np.array([0.0, 0.0, 0.0], dtype=np.float32)

    todos = []
    batches = []
    inicio = 0
    for mat, arr in dados.items():
        if not arr:
            continue
        todos.extend(arr)
        count = len(arr) // 5
        batches.append({"material": mat, "start": inicio, "count": count})
        inicio += count

    return np.array(todos, dtype=np.float32), batches, materiais, (min_v, max_v)

def skybox_vertices():
    """Esfera equiretangular para skybox — UV continuo, sem costuras."""
    def pt(phi, th):
        return (math.sin(phi)*math.cos(th), math.cos(phi), math.sin(phi)*math.sin(th))

    stacks, slices = 16, 32
    out = []
    for i in range(stacks):
        phi0, phi1 = math.pi * i / stacks, math.pi * (i+1) / stacks
        for j in range(slices):
            th0, th1 = 2*math.pi * j / slices, 2*math.pi * (j+1) / slices
            u0, u1 = j/slices, (j+1)/slices
            v0, v1 = i/stacks, (i+1)/stacks
            p00, p10 = pt(phi0, th0), pt(phi1, th0)
            p01, p11 = pt(phi0, th1), pt(phi1, th1)
            out.extend([*p00, u0, v0, *p10, u0, v1, *p11, u1, v1])
            out.extend([*p00, u0, v0, *p11, u1, v1, *p01, u1, v0])
    return np.array(out, dtype=np.float32)


## Montagem da cena e funcoes de GPU

Funcoes que conectam os dados carregados do disco com os buffers da GPU.

- `montar_cena`: orquestra `carregar_obj` + `preparar_materiais`, garantindo que todo batch
  tenha um material valido (fallback para `DEFAULT_COLOR` se ausente);
- `criar_vao_vbo`: aloca VAO e VBO, envia os vertices interleaved e configura os ponteiros
  de atributo para `posicao` (3 floats) e `texcoord` (2 floats), com stride de 5 floats;
- `desenhar_objeto`: itera sobre os batches de um objeto, ativa a textura ou a cor solida
  conforme o material, e emite a chamada `glDrawArrays`.


In [22]:
def montar_cena(caminho_obj):
    verts, batches, mats, bounds = carregar_obj(caminho_obj)
    base_dir = os.path.dirname(caminho_obj)
    materiais = preparar_materiais(mats, base_dir)

    for batch in batches:
        if batch["material"] not in materiais:
            materiais[batch["material"]] = {"cor": DEFAULT_COLOR, "tex": 0}

    return verts, batches, materiais, bounds

def criar_vao_vbo(verts, loc_pos, loc_uv):
    vao = glGenVertexArrays(1)
    glBindVertexArray(vao)

    vbo = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, verts.nbytes, verts, GL_STATIC_DRAW)

    stride = 5 * verts.itemsize
    glEnableVertexAttribArray(loc_pos)
    glVertexAttribPointer(loc_pos, 3, GL_FLOAT, False, stride, ctypes.c_void_p(0))
    glEnableVertexAttribArray(loc_uv)
    glVertexAttribPointer(loc_uv, 2, GL_FLOAT, False, stride, ctypes.c_void_p(3 * verts.itemsize))

    glBindVertexArray(0)
    return vao, vbo

def desenhar_objeto(objeto, loc_model, loc_cor, loc_usa, modelo):
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, modelo)
    glBindVertexArray(objeto["vao"])
    glActiveTexture(GL_TEXTURE0)

    for batch in objeto["batches"]:
        mat = objeto["materiais"].get(batch["material"], {"cor": DEFAULT_COLOR, "tex": 0})
        if mat["tex"] != 0:
            glUniform1i(loc_usa, 1)
            glBindTexture(GL_TEXTURE_2D, mat["tex"])
        else:
            glUniform1i(loc_usa, 0)
            glBindTexture(GL_TEXTURE_2D, 0)

        c = mat["cor"]
        glUniform4f(loc_cor, c[0], c[1], c[2], c[3])
        glDrawArrays(GL_TRIANGLES, batch["start"], batch["count"])


## Entrada de teclado e atualizacao de estado

A interacao em tempo real e dividida em duas funcoes:

`callback_tecla` — registrada como callback GLFW, mantém um conjunto das teclas atualmente
pressionadas e trata eventos pontuais de PRESS:
- `P`: alterna modo wireframe/solido;
- `L`: liga/desliga o balanco da lanterna;
- `B`: liga/desliga a translacao da bike.

`aplicar_controles` — chamada a cada frame com o delta de tempo (`dt`), atualiza:
- **Camera**: WASD movimenta no plano XZ alinhado ao yaw; Q/E sobem/descem; setas giram yaw e pitch;
- **Bike**: Z/X ajustam a escala (limitada entre 0.02 e 2.0) com velocidade suavizada por dt;
- **Lanterna**: acumula tempo e calcula `sin` para o angulo de balanco quando ativada;
- **Bike translacao**: interpola linearmente entre `bike_start_pos` e `bike_end_pos`, invertendo
  a direcao ao atingir os extremos;
- **Limites de camera**: clamp nas tres dimensoes usando a bounding box do `outside.obj` com margem de 0.5.


In [23]:
def callback_tecla(janela, key, _scancode, action, _mods):
    estado = glfw.get_window_user_pointer(janela)
    if action == glfw.PRESS:
        estado["teclas"].add(key)
        if key == glfw.KEY_P:
            estado["malha"] = not estado["malha"]
            glPolygonMode(GL_FRONT_AND_BACK, GL_LINE if estado["malha"] else GL_FILL)
        elif key == glfw.KEY_L:
            estado["lanterna_swing_on"] = not estado["lanterna_swing_on"]
        elif key == glfw.KEY_B:
            estado["bike_move"] = not estado["bike_move"]
    elif action == glfw.RELEASE:
        estado["teclas"].discard(key)

def aplicar_controles(estado, dt):
    teclas = estado["teclas"]
    vel_mov = 18.0 * dt
    vel_rot = 1.6 * dt

    yaw = estado["cam_yaw"]
    forward = np.array([math.cos(yaw), 0.0, math.sin(yaw)], dtype=np.float32)
    right = np.array([-math.sin(yaw), 0.0, math.cos(yaw)], dtype=np.float32)

    mov = np.zeros(3, dtype=np.float32)
    mov += forward * ((glfw.KEY_W in teclas) - (glfw.KEY_S in teclas))
    mov += right * ((glfw.KEY_D in teclas) - (glfw.KEY_A in teclas))
    mov[1] += ((glfw.KEY_E in teclas) - (glfw.KEY_Q in teclas))
    estado["cam_pos"] += mov * vel_mov

    estado["cam_yaw"] += ((glfw.KEY_RIGHT in teclas) - (glfw.KEY_LEFT in teclas)) * vel_rot
    estado["cam_pitch"] += ((glfw.KEY_UP in teclas) - (glfw.KEY_DOWN in teclas)) * vel_rot
    estado["cam_pitch"] = float(np.clip(estado["cam_pitch"], -1.2, 1.2))

    esc_delta = ((glfw.KEY_Z in teclas) - (glfw.KEY_X in teclas)) * dt * 0.3
    if esc_delta != 0.0:
        estado["bike_scale"] += esc_delta
        estado["bike_scale"] = float(np.clip(estado["bike_scale"], 0.02, 2.0))

    if estado["lanterna_swing_on"]:
        estado["lanterna_swing_time"] += dt
        estado["lanterna_swing"] = math.sin(estado["lanterna_swing_time"] * 2.0) * 0.5
    else:
        estado["lanterna_swing"] = 0.0

    if estado["bike_move"]:
        estado["bike_t"] += estado["bike_dir"] * estado["bike_speed"] * dt
        if estado["bike_t"] > 1.0:
            estado["bike_t"] = 1.0
            estado["bike_dir"] = -1.0
        elif estado["bike_t"] < 0.0:
            estado["bike_t"] = 0.0
            estado["bike_dir"] = 1.0

    min_v, max_v = estado["bounds"]
    margem = 0.5
    estado["cam_pos"][0] = float(np.clip(estado["cam_pos"][0], min_v[0] - margem, max_v[0] + margem))
    estado["cam_pos"][2] = float(np.clip(estado["cam_pos"][2], min_v[2] - margem, max_v[2] + margem))
    estado["cam_pos"][1] = float(np.clip(estado["cam_pos"][1], min_v[1] + 0.5, max_v[1] + 500.0))


## Funcao principal e render loop

`criar_programa` compila e linka os shaders GLSL, verificando erros em cada etapa.

`main` inicializa o contexto e executa o loop principal:

1. **Setup**: cria janela GLFW 1280x720, compila shaders, carrega os tres objetos (`outside.obj`,
   `lanterna/lantern_obj.obj`, `bike/11717_bicycle_v2_L1.obj`) e a textura de skybox;
2. **Camera inicial**: posicionada em `bike_start_pos`, orientada para o centro da cena via `look_at`;
3. **Skybox**: desenhada primeiro com depth write desativado (`glDepthMask(GL_FALSE)`) e a componente
   de translacao da matriz de visao zerada, para que acompanhe apenas a rotacao da camera.
   Escalonada para 200 unidades e renderizada como esfera equiretangular;
4. **Objetos**: `outside` com matriz identidade; `lanterna` com translacao + rotacao de balanco + escala;
   `bike` com interpolacao de posicao + yaw fixo + tilt de -90 graus + escala interativa;
5. **Cleanup**: libera VAOs, VBOs, texturas e o programa de shader ao fechar a janela.


In [24]:
def criar_programa(cod_vs, cod_fs):
    def compilar(codigo, tipo):
        shader = glCreateShader(tipo)
        glShaderSource(shader, codigo)
        glCompileShader(shader)
        if not glGetShaderiv(shader, GL_COMPILE_STATUS):
            raise RuntimeError(glGetShaderInfoLog(shader).decode("utf-8"))
        return shader

    vs = compilar(cod_vs, GL_VERTEX_SHADER)
    fs = compilar(cod_fs, GL_FRAGMENT_SHADER)
    prog = glCreateProgram()
    glAttachShader(prog, vs)
    glAttachShader(prog, fs)
    glLinkProgram(prog)

    if not glGetProgramiv(prog, GL_LINK_STATUS):
        raise RuntimeError(glGetProgramInfoLog(prog).decode("utf-8"))

    glDeleteShader(vs)
    glDeleteShader(fs)
    return prog

def main():
    if not glfw.init():
        raise RuntimeError("Erro ao inicializar GLFW")

    janela = glfw.create_window(1280, 720, "Projeto2 - Outside", None, None)
    if not janela:
        glfw.terminate()
        raise RuntimeError("Erro ao criar janela")

    glfw.make_context_current(janela)

    estado = {
        "malha": False,
        "teclas": set(),
        "cam_pos": np.zeros(3, dtype=np.float32),
        "cam_yaw": 0.0,
        "cam_pitch": 0.0,
        "bounds": (np.array([-1.0, -1.0, -1.0], dtype=np.float32), np.array([1.0, 1.0, 1.0], dtype=np.float32)),
        "lanterna_scale": 0.2,
        "lanterna_base_scale": 0.05,
        "lanterna_swing": 0.0,
        "lanterna_swing_time": 0.0,
        "lanterna_swing_on": False,
        "bike_t": 0.0,
        "bike_dir": 1.0,
        "bike_speed": 0.12,
        "bike_move": False,
        "bike_scale": 0.1,
        "bike_tilt": -math.pi / 2.0,
        "bike_yaw_offset": -math.pi / 2.0,
        "lanterna_pos": np.array([33, 5.3, 107.064], dtype=np.float32),
        "bike_start_pos": np.array([50, 0.644107, 100], dtype=np.float32),
        "bike_end_pos": np.array([15, 0.644107, 100], dtype=np.float32),
    }
    glfw.set_window_user_pointer(janela, estado)
    glfw.set_key_callback(janela, callback_tecla)

    prog = criar_programa(SHADER_VS, SHADER_FS)
    glUseProgram(prog)

    loc_model = glGetUniformLocation(prog, "modelo")
    loc_view = glGetUniformLocation(prog, "visao")
    loc_proj = glGetUniformLocation(prog, "projecao")
    loc_cor = glGetUniformLocation(prog, "cor")
    loc_usa = glGetUniformLocation(prog, "usa_textura")

    loc_pos = glGetAttribLocation(prog, "posicao")
    loc_uv = glGetAttribLocation(prog, "texcoord")

    base_dir = os.getcwd()
    outside_path = os.path.join(base_dir, "outside.obj")
    lanterna_path = os.path.join(base_dir, "lanterna", "lantern_obj.obj")
    bike_path = os.path.join(base_dir, "bike", "11717_bicycle_v2_L1.obj")
    skybox_tex_path = os.path.join(base_dir, "Simple_Sky-Blue_04-512x512.png")
    if not os.path.exists(outside_path):
        raise RuntimeError("Arquivo outside.obj nao encontrado no diretorio atual")
    if not os.path.exists(lanterna_path):
        raise RuntimeError("Arquivo lanterna/lantern_obj.obj nao encontrado no diretorio atual")
    if not os.path.exists(bike_path):
        raise RuntimeError("Arquivo bike/11717_bicycle_v2_L1.obj nao encontrado no diretorio atual")
    if not os.path.exists(skybox_tex_path):
        raise RuntimeError("Arquivo Simple_Sky-Blue_04-512x512.png nao encontrado no diretorio atual")

    outside_verts, outside_batches, outside_materiais, bounds = montar_cena(outside_path)
    if outside_verts.size == 0:
        raise RuntimeError("Nenhum vertice carregado de outside.obj")
    outside_vao, outside_vbo = criar_vao_vbo(outside_verts, loc_pos, loc_uv)
    outside = {"vao": outside_vao, "vbo": outside_vbo, "batches": outside_batches, "materiais": outside_materiais}

    lanterna_verts, lanterna_batches, lanterna_materiais, _ = montar_cena(lanterna_path)
    lanterna_vao, lanterna_vbo = criar_vao_vbo(lanterna_verts, loc_pos, loc_uv)
    lanterna = {"vao": lanterna_vao, "vbo": lanterna_vbo, "batches": lanterna_batches, "materiais": lanterna_materiais}

    bike_verts, bike_batches, bike_materiais, _ = montar_cena(bike_path)
    bike_vao, bike_vbo = criar_vao_vbo(bike_verts, loc_pos, loc_uv)
    bike = {"vao": bike_vao, "vbo": bike_vbo, "batches": bike_batches, "materiais": bike_materiais}

    skybox_verts = skybox_vertices()
    skybox_vao, skybox_vbo = criar_vao_vbo(skybox_verts, loc_pos, loc_uv)
    skybox_tex = carregar_textura(skybox_tex_path, clamp=True)
    if skybox_tex == 0:
        raise RuntimeError("Falha ao carregar a textura Simple_Sky-Blue_04-512x512.png")


    objetos = [outside, lanterna, bike]
    estado["bounds"] = bounds
    centro = (bounds[0] + bounds[1]) * 0.5

    estado["cam_pos"] = estado["bike_start_pos"].copy()

    dir_ini = centro - estado["cam_pos"]
    dir_ini_norm = dir_ini / np.linalg.norm(dir_ini)
    estado["cam_yaw"] = math.atan2(dir_ini_norm[0], dir_ini_norm[2])
    estado["cam_pitch"] = math.asin(float(np.clip(dir_ini_norm[1], -1.0, 1.0)))

    glEnable(GL_DEPTH_TEST)
    glClearColor(0.65, 0.75, 0.90, 1.0)


    bike_dir = estado["bike_end_pos"] - estado["bike_start_pos"]
    if np.linalg.norm(bike_dir) > 0.0:
        bike_yaw = math.atan2(bike_dir[0], bike_dir[2]) + estado["bike_yaw_offset"]
    else:
        bike_yaw = estado["bike_yaw_offset"]

    tempo_ant = glfw.get_time()
    while not glfw.window_should_close(janela):
        agora = glfw.get_time()
        dt = agora - tempo_ant
        tempo_ant = agora

        glfw.poll_events()
        aplicar_controles(estado, dt)

        larg, alt = glfw.get_framebuffer_size(janela)
        asp = larg / max(alt, 1)
        proj = perspectiva(math.radians(60.0), asp, 0.1, 200.0)

        yaw, pitch = estado["cam_yaw"], estado["cam_pitch"]
        cam_dir = np.array([
            math.cos(pitch) * math.cos(yaw),
            math.sin(pitch),
            math.cos(pitch) * math.sin(yaw),
        ], dtype=np.float32)
        visao = look_at(estado["cam_pos"], estado["cam_pos"] + cam_dir)

        glViewport(0, 0, larg, alt)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

        glUniformMatrix4fv(loc_proj, 1, GL_TRUE, proj)

        glDepthMask(GL_FALSE)
        glDisable(GL_DEPTH_TEST)
        view_sky = visao.copy()
        view_sky[0, 3] = 0.0
        view_sky[1, 3] = 0.0
        view_sky[2, 3] = 0.0
        glUniformMatrix4fv(loc_view, 1, GL_TRUE, view_sky)
        glUniform1i(loc_usa, 1)
        glBindTexture(GL_TEXTURE_2D, skybox_tex)
        glUniform4f(loc_cor, 1.0, 1.0, 1.0, 1.0)
        glUniformMatrix4fv(loc_model, 1, GL_TRUE, esc(200.0, 200.0, 200.0))
        glBindVertexArray(skybox_vao)
        glDrawArrays(GL_TRIANGLES, 0, len(skybox_verts) // 5)
        glBindVertexArray(0)
        glBindTexture(GL_TEXTURE_2D, 0)
        glDepthMask(GL_TRUE)
        glEnable(GL_DEPTH_TEST)

        glUniformMatrix4fv(loc_view, 1, GL_TRUE, visao)

        mat_outside = ident()
        desenhar_objeto(outside, loc_model, loc_cor, loc_usa, mat_outside)

        lanterna_pos = estado["lanterna_pos"]
        s = estado["lanterna_scale"] * estado["lanterna_base_scale"]
        swing = estado["lanterna_swing"]
        mat_lanterna = transl(lanterna_pos[0], lanterna_pos[1], lanterna_pos[2]) @ rot_x(swing) @ esc(s, s, s)
        desenhar_objeto(lanterna, loc_model, loc_cor, loc_usa, mat_lanterna)

        t = estado["bike_t"]
        bike_start = estado["bike_start_pos"]
        bike_end = estado["bike_end_pos"]
        bike_pos = bike_start * (1.0 - t) + bike_end * t
        bscale = estado["bike_scale"]
        tilt = estado["bike_tilt"]
        mat_bike = transl(bike_pos[0], bike_pos[1], bike_pos[2]) @ rot_y(bike_yaw) @ rot_x(tilt) @ esc(bscale, bscale, bscale)
        desenhar_objeto(bike, loc_model, loc_cor, loc_usa, mat_bike)

        glfw.swap_buffers(janela)

    for obj in objetos:
        glDeleteBuffers(1, [obj["vbo"]])
        glDeleteVertexArrays(1, [obj["vao"]])
    glDeleteBuffers(1, [skybox_vbo])
    glDeleteVertexArrays(1, [skybox_vao])
    if skybox_tex:
        glDeleteTextures(1, [skybox_tex])
    glDeleteProgram(prog)
    for obj in objetos:
        for mat in obj["materiais"].values():
            if mat["tex"]:
                glDeleteTextures(1, [mat["tex"]])
    glfw.terminate()

main()


## Controles

| Tecla | Acao |
|-------|------|
| W / S | Mover camera para frente / tras |
| A / D | Mover camera para esquerda / direita |
| Q / E | Mover camera para baixo / cima |
| Setas | Rotacionar camera (yaw e pitch) |
| Z / X | Diminuir / aumentar escala da bike |
| B | Liga/desliga translacao da bike |
| L | Liga/desliga balanco da lanterna |
| P | Alterna modo solido / wireframe |
